# GOV-01 V2-E7 — EfficientNetB0 transfer learning

**Purpose:** test a different pretrained visual model, EfficientNetB0, on the unchanged V2 road-condition task. This model cannot reuse MobileNetV2 weights because its internal design is different; it starts from EfficientNetB0's ImageNet knowledge.

Training happens in two safe stages: first train a new seven-class classifier head while the EfficientNetB0 visual base is frozen; then carefully fine-tune only its upper layers. The approved extra attention for **crack**, **pothole**, and **repaired road** is retained.

- Only train and validation are used. The protected test is deliberately never loaded.
- No dataset, image, label, or split is changed.
- V2-E7 saves its own checkpoints in Drive and never overwrites MobileNetV2 experiments.

In [ ]:
# Cell 1 — imports and Google Drive connection
from pathlib import Path
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive, files
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support

drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# Cell 2 — fixed paths and approved V2 class order
DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
ARCHIVE_PATH = DRIVE_ROOT / 'v2' / 'multiclass_final.zip'
PERSISTENT_DIR = DRIVE_ROOT / 'v2_e7_checkpoint'

DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e7_output')
HEAD_CHECKPOINT_PATH = PERSISTENT_DIR / 'v2_e7_efficientnetb0_frozen_head_best.keras'
HEAD_HISTORY_PATH = PERSISTENT_DIR / 'v2_e7_frozen_head_history.csv'
HEAD_SUMMARY_PATH = PERSISTENT_DIR / 'v2_e7_frozen_head_summary.json'
CHECKPOINT_PATH = PERSISTENT_DIR / 'v2_e7_efficientnetb0_finetuned_best.keras'
FINE_HISTORY_PATH = PERSISTENT_DIR / 'v2_e7_finetuned_history.csv'
SUMMARY_PATH = PERSISTENT_DIR / 'v2_e7_training_summary.json'

CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
TARGET_CLASS_WEIGHTS = {'crack': 1.20, 'pothole': 1.60, 'repaired_road': 1.60}
class_weight = {index: TARGET_CLASS_WEIGHTS.get(class_name, 1.00) for index, class_name in enumerate(CLASS_NAMES)}

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Missing V2 dataset ZIP in Google Drive: {ARCHIVE_PATH}')
print('Dataset ZIP:', ARCHIVE_PATH)
print('V2-E7 recovery folder:', PERSISTENT_DIR)
print('Class priorities:', {CLASS_NAMES[index]: weight for index, weight in class_weight.items()})

In [ ]:
# Cell 3 — extract the persistent dataset into the temporary Colab runtime only when needed.
if not (DATA_DIR / 'train').is_dir() or not (DATA_DIR / 'validation').is_dir():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted dataset to:', DATA_DIR)
else:
    print('Dataset already available in this runtime:', DATA_DIR)

for split_name, expected_count in [('train', 9452), ('validation', 1658)]:
    split_path = DATA_DIR / split_name
    if not split_path.is_dir():
        raise FileNotFoundError(f'Missing required split: {split_path}')
    actual_count = sum(1 for path in split_path.rglob('*') if path.is_file())
    if actual_count != expected_count:
        raise RuntimeError(f'{split_name} has {actual_count} files; expected {expected_count}. Stop and inspect the archive.')
    print(f'{split_name}: {actual_count} images')

if (DATA_DIR / 'protected_test').exists():
    print('Protected test is present in the archive but is intentionally not loaded or evaluated.')

In [ ]:
# Cell 4 — same approved V2 train and validation loaders.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', label_mode='int', class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', label_mode='int', class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

if train_ds.class_names != CLASS_NAMES or validation_ds.class_names != CLASS_NAMES:
    raise RuntimeError('Class order differs from the approved V2 label order.')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)
print('Class order:', CLASS_NAMES)
print('Protected test remains unused.')

In [ ]:
# Cell 5 — two-stage EfficientNetB0 training. Run once; it resumes safely from completed stages.
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not HEAD_SUMMARY_PATH.is_file():
    inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name='image')
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet', input_tensor=inputs
    )
    base_model.trainable = False
    features = base_model(inputs, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling')(features)
    features = tf.keras.layers.Dropout(0.30, name='dropout')(features)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='road_condition')(features)
    model = tf.keras.Model(inputs, outputs, name='v2_e7_efficientnetb0')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('Stage A: frozen EfficientNetB0 visual base; training the new road-condition classifier head.')
    stage_a_start = time.time()
    stage_a_history = model.fit(
        train_ds, validation_data=validation_ds, epochs=12, class_weight=class_weight, verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(HEAD_CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
            tf.keras.callbacks.CSVLogger(HEAD_HISTORY_PATH, append=False),
        ],
    )
    head_summary = {
        'stage': 'frozen_head', 'epochs_completed': len(stage_a_history.history['loss']),
        'training_seconds': time.time() - stage_a_start,
        'best_validation_loss': float(min(stage_a_history.history['val_loss'])),
        'best_validation_accuracy': float(max(stage_a_history.history['val_accuracy'])),
    }
    HEAD_SUMMARY_PATH.write_text(json.dumps(head_summary, indent=2), encoding='utf-8')
    print('Stage A complete. Saved:', HEAD_CHECKPOINT_PATH)
else:
    print('Stage A is already complete; reusing its saved best checkpoint.')

if not SUMMARY_PATH.is_file():
    if not HEAD_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f'Missing completed Stage A checkpoint: {HEAD_CHECKPOINT_PATH}')
    model = tf.keras.models.load_model(HEAD_CHECKPOINT_PATH)
    base_candidates = [layer for layer in model.layers if isinstance(layer, tf.keras.Model) and layer.name.startswith('efficientnetb0')]
    if len(base_candidates) != 1:
        raise RuntimeError(f'Expected one EfficientNetB0 base model, found {len(base_candidates)}.')
    base_model = base_candidates[0]
    UNFREEZE_LAST_LAYERS = 30
    fine_tune_start = max(0, len(base_model.layers) - UNFREEZE_LAST_LAYERS)
    base_model.trainable = True
    for layer in base_model.layers[:fine_tune_start]:
        layer.trainable = False
    for layer in base_model.layers[fine_tune_start:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f'Stage B: fine-tuning the upper {UNFREEZE_LAST_LAYERS} EfficientNetB0 layers; batch-normalization layers stay frozen.')
    stage_b_start = time.time()
    stage_b_history = model.fit(
        train_ds, validation_data=validation_ds, epochs=15, class_weight=class_weight, verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
            tf.keras.callbacks.CSVLogger(FINE_HISTORY_PATH, append=False),
        ],
    )
    training_summary = {
        'experiment': 'V2-E7', 'backbone': 'EfficientNetB0', 'starting_weights': 'ImageNet',
        'target_class_weights': TARGET_CLASS_WEIGHTS, 'unfreeze_last_layers': UNFREEZE_LAST_LAYERS,
        'stage_b_epochs_completed': len(stage_b_history.history['loss']),
        'stage_b_training_seconds': time.time() - stage_b_start,
        'best_validation_loss': float(min(stage_b_history.history['val_loss'])),
        'best_validation_accuracy': float(max(stage_b_history.history['val_accuracy'])),
    }
    SUMMARY_PATH.write_text(json.dumps(training_summary, indent=2), encoding='utf-8')
    print('Stage B complete. Best V2-E7 model saved:', CHECKPOINT_PATH)
else:
    print('Stage B is already complete; do not retrain. Continue to Cell 6 for validation results.')

In [ ]:
# Cell 6 — validate the saved best V2-E7 model. After disconnect, rerun Cells 1–4 before this cell.
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'No completed V2-E7 fine-tuned checkpoint found in Drive: {CHECKPOINT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model = tf.keras.models.load_model(CHECKPOINT_PATH)
y_validation = np.concatenate([labels.numpy() for _, labels in validation_ds])
probabilities = best_model.predict(validation_ds, verbose=1)
predictions = probabilities.argmax(axis=1)

precision, recall, f1_values, support = precision_recall_fscore_support(
    y_validation, predictions, labels=np.arange(len(CLASS_NAMES)), zero_division=0
)
e7_metrics = {
    'experiment': 'V2-E7', 'backbone': 'EfficientNetB0', 'split': 'validation',
    'protected_test_used': False, 'image_count': int(len(y_validation)),
    'accuracy': float(accuracy_score(y_validation, predictions)),
    'macro_f1': float(f1_score(y_validation, predictions, average='macro')),
    'per_class': {class_name: {'precision': float(precision[index]), 'recall': float(recall[index]), 'f1': float(f1_values[index]), 'support': int(support[index])} for index, class_name in enumerate(CLASS_NAMES)},
}
(OUTPUT_DIR / 'v2_e7_validation_metrics.json').write_text(json.dumps(e7_metrics, indent=2), encoding='utf-8')
comparison = {
    'split': 'validation', 'protected_test_used': False,
    'V2-E1_compact_cnn': {'accuracy': 0.6984318455971049, 'macro_f1': 0.6795904943261485},
    'V2-E2_frozen_mobilenetv2': {'accuracy': 0.8196622436670687, 'macro_f1': 0.8374032394969715},
    'V2-E3_finetuned_mobilenetv2': {'accuracy': 0.8552472858866104, 'macro_f1': 0.8704102628125944},
    'V2-E4_targeted_class_weights': {'accuracy': 0.8642943305186972, 'macro_f1': 0.8803169234751674},
    'V2-E7_efficientnetb0': {'accuracy': e7_metrics['accuracy'], 'macro_f1': e7_metrics['macro_f1']},
}
(OUTPUT_DIR / 'v2_e7_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
report = classification_report(y_validation, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0)
(OUTPUT_DIR / 'v2_e7_validation_classification_report.txt').write_text(report, encoding='utf-8')
print('Validation accuracy:', f"{e7_metrics['accuracy']:.4f}")
print('Validation macro F1:', f"{e7_metrics['macro_f1']:.4f}")
print(report)
matrix = confusion_matrix(y_validation, predictions, labels=np.arange(len(CLASS_NAMES)))
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(matrix, cmap='Blues')
axis.set_xticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
axis.set_yticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES)
axis.set_xlabel('Predicted label'); axis.set_ylabel('Actual label'); axis.set_title('V2-E7 validation confusion matrix')
threshold = matrix.max() / 2 if matrix.max() else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, str(matrix[row, column]), ha='center', va='center', color='white' if matrix[row, column] > threshold else 'black')
fig.colorbar(image, ax=axis); fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'v2_e7_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Cell 7 — learning curves and browser-download package. Do not commit the package or model.
if not HEAD_HISTORY_PATH.is_file() or not FINE_HISTORY_PATH.is_file():
    raise FileNotFoundError('Both Stage A and Stage B histories are required before creating the V2-E7 package.')

head_history = pd.read_csv(HEAD_HISTORY_PATH)
fine_history = pd.read_csv(FINE_HISTORY_PATH)
fine_history['epoch'] = fine_history['epoch'] + len(head_history)
history_frame = pd.concat([head_history, fine_history], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_frame['epoch'] + 1, history_frame['accuracy'], label='train')
axes[0].plot(history_frame['epoch'] + 1, history_frame['val_accuracy'], label='validation')
axes[0].axvline(len(head_history), color='gray', linestyle='--', label='fine-tuning starts')
axes[0].set_title('V2-E7 accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].plot(history_frame['epoch'] + 1, history_frame['loss'], label='train')
axes[1].plot(history_frame['epoch'] + 1, history_frame['val_loss'], label='validation')
axes[1].axvline(len(head_history), color='gray', linestyle='--', label='fine-tuning starts')
axes[1].set_title('V2-E7 loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'v2_e7_learning_curve.png', dpi=150); plt.show()

for source_path in [CHECKPOINT_PATH, HEAD_HISTORY_PATH, FINE_HISTORY_PATH, HEAD_SUMMARY_PATH, SUMMARY_PATH]:
    if source_path.is_file():
        shutil.copy2(source_path, OUTPUT_DIR / source_path.name)
archive_path = shutil.make_archive('/content/v2_e7_output', 'zip', OUTPUT_DIR)
print('Created browser-download package:', archive_path)
print('Keep this ZIP outside the Git repository. Do not commit it.')
files.download(archive_path)